# AION-U1: Adapter gegen Baseline messen

Gleiche Suite, gleiche Aggregation — nur der Adapter unterscheidet sich.
Die Basis ist `benchmark_hard_text_v1.yaml` (24 Faelle, 12 Capabilities);
Qwen2.5-1.5B-Instruct ist text-only, die zwei Bild-Faecke sind daher
entfernt. `--max-tokens 1024` verhindert Abbrueche bei langen Antworten.

In [ ]:
# 1) Repo und CLI
!git clone -q https://github.com/Reznap87/aion-u1.git
!cd /content/aion-u1 && pip -q install -e .
import pathlib
repo = pathlib.Path('/content/aion-u1')
print('Bereit:', (repo/'training/configs/sft_lora_open_v3.yaml').is_file())


In [ ]:
# 2) Trainierten Adapter hochladen (sft_pilot_v3_ergebnis.zip oder der neue Colab-Zip)
from google.colab import files
import zipfile, pathlib
hochgeladen = files.upload()
zip_pfad = next(pathlib.Path(p) for p in hochgeladen)
ziel = pathlib.Path('/content/adapter')
ziel.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_pfad) as z:
    z.extractall(ziel)
treffer = sorted(ziel.rglob('adapter_model.safetensors'))
assert treffer, 'Kein adapter_model.safetensors im ZIP gefunden'
adapter = str(treffer[0].parent)
print('Adapter:', adapter)


In [ ]:
# 3) Baseline — dasselbe Modell ohne Adapter
!cd /content/aion-u1 && aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct --suite evaluations/capability_suite/benchmark_hard_text_v1.yaml --device cuda --max-tokens 1024 --output reports/eval-baseline.json


In [ ]:
# 4) Adapter — exakt dieselbe Suite und Aggregation
!cd /content/aion-u1 && aion-u1-evaluate-model --base-model Qwen/Qwen2.5-1.5B-Instruct --adapter {adapter} --suite evaluations/capability_suite/benchmark_hard_text_v1.yaml --device cuda --max-tokens 1024 --output reports/eval-adapter.json


In [ ]:
# 5) Vergleich und Download
import json, pathlib
from google.colab import files
r = pathlib.Path('/content/aion-u1/reports')
b = json.loads((r/'eval-baseline.json').read_text(encoding='utf-8'))
a = json.loads((r/'eval-adapter.json').read_text(encoding='utf-8'))
print('Baseline:', json.dumps(b.get('scores', {}), ensure_ascii=False)[:400])
print('Adapter :', json.dumps(a.get('scores', {}), ensure_ascii=False)[:400])
for n in ('eval-baseline.json', 'eval-adapter.json'):
    files.download(str(r/n))
